### Импорты и загрузка данных

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!apt-get install unrar
!unrar x origs_NER.rar
!unrar x trans_NER.rar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from origs_NER.rar

Creating    content                                                   OK
Creating    content/origs_NER                                         OK
Extracting  content/origs_NER/130403.txt                                   0%  OK 
Extracting  content/origs_NER/81070.txt                                    0%  OK 
Extracting  content/origs_NER/275073.txt                                   0%  OK 
Extracting  content/origs_NER/266250.txt                                   0%  OK 
Extracting  content/origs_NER/67789.txt                                    0%  OK 
Extracting  content/origs_NER/49812.txt                             

In [ ]:
def load_texts(folder, label):
    texts, labels = [], []
    for file in os.listdir(folder):
        if file.endswith(".txt"):
            with open(os.path.join(folder, file), encoding="utf-8") as f:
                text = f.read()
                text = text.split('\n', 1)[-1]
                texts.append(text)
                labels.append(label)
    return texts, labels

In [ ]:
orig_texts, orig_labels = load_texts("/content/content/origs_NER", 0)
trans_texts, trans_labels = load_texts("/content/content/trans_NER", 1)

In [ ]:
texts = orig_texts + trans_texts
labels = orig_labels + trans_labels

In [ ]:
df = pd.DataFrame({"text": texts, "label": labels})
print(f"Датасет: {len(df)} текстов, оригиналов: {sum(l==0 for l in labels)}, переводов: {sum(l==1 for l in labels)}")

Датасет: 1710 текстов, оригиналов: 855, переводов: 855


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print(f"Train: {len(train_texts)}, Test: {len(test_texts)}")

Train: 1368, Test: 342


In [ ]:
def evaluate(name, y_true, y_pred):
    """
    Выводит accuracy, F1 (macro и binary) и classification report.
    Возвращает словарь с метриками для дальнейшего сравнения.
    """
    acc = accuracy_score(y_true, y_pred)
    f1_bin = f1_score(y_true, y_pred, average="binary")
    f1_mac = f1_score(y_true, y_pred, average="macro")

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1 binary: {f1_bin:.4f}  (класс 1 = перевод)")
    print(f"  F1 macro : {f1_mac:.4f}  (среднее по обоим классам)")
    print("\nClassification report:")
    print(classification_report(y_true, y_pred, target_names=["оригинал", "перевод"]))
    print("Confusion matrix (строки = истина, столбцы = предсказание):")
    print(confusion_matrix(y_true, y_pred))

    return {"model": name, "accuracy": acc, "f1_binary": f1_bin, "f1_macro": f1_mac}

### Модель 1: Логистическая регрессия

In [ ]:
lr_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            max_features=100_000,
            min_df=2,
            sublinear_tf=True,
        )
    ),
    (
        "clf",
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42,
            n_jobs=-1
        )
    )
])

lr_pipeline.fit(train_texts, train_labels)
lr_preds = lr_pipeline.predict(test_texts)
lr_results = evaluate("TF-IDF + Logistic Regression", test_labels, lr_preds)


  TF-IDF + Logistic Regression
  Accuracy : 0.9327
  F1 binary: 0.9283  (класс 1 = перевод)
  F1 macro : 0.9325  (среднее по обоим классам)

Classification report:
              precision    recall  f1-score   support

    оригинал       0.89      0.99      0.94       171
     перевод       0.99      0.87      0.93       171

    accuracy                           0.93       342
   macro avg       0.94      0.93      0.93       342
weighted avg       0.94      0.93      0.93       342

Confusion matrix (строки = истина, столбцы = предсказание):
[[170   1]
 [ 22 149]]


### Модель 2: LinearSVC

In [ ]:
svc_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            max_features=100_000,
            min_df=2,
            sublinear_tf=True,
        )
    ),
    (
        "clf",
        LinearSVC(
            C=1.0,
            max_iter=2000,
            random_state=42
        )
    )
])

svc_pipeline.fit(train_texts, train_labels)
svc_preds = svc_pipeline.predict(test_texts)
svc_results = evaluate("TF-IDF + LinearSVC", test_labels, svc_preds)


  TF-IDF + LinearSVC
  Accuracy : 0.9649
  F1 binary: 0.9636  (класс 1 = перевод)
  F1 macro : 0.9649  (среднее по обоим классам)

Classification report:
              precision    recall  f1-score   support

    оригинал       0.93      1.00      0.97       171
     перевод       1.00      0.93      0.96       171

    accuracy                           0.96       342
   macro avg       0.97      0.96      0.96       342
weighted avg       0.97      0.96      0.96       342

Confusion matrix (строки = истина, столбцы = предсказание):
[[171   0]
 [ 12 159]]


### Модель 3: символьные n-граммы (попытаемся учесть русскую морфологию)

In [ ]:
char_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            max_features=200_000,
            min_df=3,
            sublinear_tf=True,
        )
    ),
    (
        "clf",
        LinearSVC(C=1.0, max_iter=2000, random_state=42)
    )
])

char_pipeline.fit(train_texts, train_labels)
char_preds = char_pipeline.predict(test_texts)
char_results = evaluate("Char n-grams (3-5) + LinearSVC", test_labels, char_preds)


  Char n-grams (3-5) + LinearSVC
  Accuracy : 0.9649
  F1 binary: 0.9636  (класс 1 = перевод)
  F1 macro : 0.9649  (среднее по обоим классам)

Classification report:
              precision    recall  f1-score   support

    оригинал       0.93      1.00      0.97       171
     перевод       1.00      0.93      0.96       171

    accuracy                           0.96       342
   macro avg       0.97      0.96      0.96       342
weighted avg       0.97      0.96      0.96       342

Confusion matrix (строки = истина, столбцы = предсказание):
[[171   0]
 [ 12 159]]


### Сводная таблица результатов

In [ ]:
results_df = pd.DataFrame([lr_results, svc_results, char_results])
results_df = results_df.set_index("model").round(4)

print(results_df.to_string())

                                accuracy  f1_binary  f1_macro
model                                                        
TF-IDF + Logistic Regression      0.9327     0.9283    0.9325
TF-IDF + LinearSVC                0.9649     0.9636    0.9649
Char n-grams (3-5) + LinearSVC    0.9649     0.9636    0.9649


Топ-20 признаков для лог рег

In [ ]:
vectorizer = lr_pipeline.named_steps["tfidf"]
classifier = lr_pipeline.named_steps["clf"]
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = classifier.coef_[0]

top_n = 20
top_translation = feature_names[np.argsort(coefs)[-top_n:][::-1]]
top_original = feature_names[np.argsort(coefs)[:top_n]]

print(f"Маркеры перевода:")
for i, feat in enumerate(top_translation, 1):
    print(f"  {i:2d}. {feat:30s}  coef={coefs[np.where(feature_names==feat)[0][0]]:+.4f}")

print(f"\nМаркеры оригинала:")
for i, feat in enumerate(top_original, 1):
    print(f"  {i:2d}. {feat:30s}  coef={coefs[np.where(feature_names==feat)[0][0]]:+.4f}")

Маркеры перевода:
   1. прежде чем                      coef=+1.2433
   2. прежде                          coef=+1.1455
   3. когда он                        coef=+0.8561
   4. затем                           coef=+0.8391
   5. чтобы                           coef=+0.8307
   6. поскольку                       coef=+0.7646
   7. когда либо                      coef=+0.7217
   8. все еще                         coef=+0.7027
   9. когда она                       coef=+0.6546
  10. когда                           coef=+0.6424
  11. конце концов                    coef=+0.6417
  12. должно быть                     coef=+0.6286
  13. вероятно                        coef=+0.6227
  14. концов                          coef=+0.6168
  15. тех пор                         coef=+0.6110
  16. сказал он                       coef=+0.6102
  17. пор как                         coef=+0.6086
  18. никогда не                      coef=+0.6079
  19. крайней мере                    coef=+0.5902
  20. по край

Топ-20 признаков для LinearSVC

In [ ]:
vectorizer = svc_pipeline.named_steps["tfidf"]
classifier = svc_pipeline.named_steps["clf"]
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = classifier.coef_[0]

top_n = 20
top_translation = feature_names[np.argsort(coefs)[-top_n:][::-1]]
top_original = feature_names[np.argsort(coefs)[:top_n]]

print(f"Маркеры перевода:")
for i, feat in enumerate(top_translation, 1):
    print(f"  {i:2d}. {feat:30s}  coef={coefs[np.where(feature_names==feat)[0][0]]:+.4f}")

print(f"\nМаркеры оригинала:")
for i, feat in enumerate(top_original, 1):
    print(f"  {i:2d}. {feat:30s}  coef={coefs[np.where(feature_names==feat)[0][0]]:+.4f}")

Маркеры перевода:
   1. прежде чем                      coef=+0.9786
   2. прежде                          coef=+0.9136
   3. чтобы                           coef=+0.8207
   4. затем                           coef=+0.6751
   5. когда он                        coef=+0.6509
   6. когда                           coef=+0.6436
   7. перевод                         coef=+0.6298
   8. поскольку                       coef=+0.6273
   9. его                             coef=+0.5881
  10. переводчика                     coef=+0.5758
  11. никогда не                      coef=+0.5756
  12. когда либо                      coef=+0.5676
  13. никогда                         coef=+0.5606
  14. конце концов                    coef=+0.5379
  15. он                              coef=+0.5259
  16. должно быть                     coef=+0.5117
  17. концов                          coef=+0.5101
  18. чем                             coef=+0.5093
  19. все еще                         coef=+0.5037
  20. быть   

Топ-20 признаков для n-грамм

In [ ]:
vectorizer = char_pipeline.named_steps["tfidf"]
classifier = char_pipeline.named_steps["clf"]
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = classifier.coef_[0]

top_n = 20
top_translation = feature_names[np.argsort(coefs)[-top_n:][::-1]]
top_original = feature_names[np.argsort(coefs)[:top_n]]

print(f"Маркеры перевода:")
for i, feat in enumerate(top_translation, 1):
    print(f"  {i:2d}. {feat:30s}  coef={coefs[np.where(feature_names==feat)[0][0]]:+.4f}")

print(f"\nМаркеры оригинала:")
for i, feat in enumerate(top_original, 1):
    print(f"  {i:2d}. {feat:30s}  coef={coefs[np.where(feature_names==feat)[0][0]]:+.4f}")

Маркеры перевода:
   1. жде                             coef=+0.4656
   2. ежде                            coef=+0.4647
   3. режде                           coef=+0.4413
   4.  преж                           coef=+0.4240
   5. прежд                           coef=+0.4223
   6. ежде                            coef=+0.4149
   7. режд                            coef=+0.4135
   8. преж                            coef=+0.4102
   9. борм                            coef=+0.3877
  10. бормо                           coef=+0.3854
  11. ревод                           coef=+0.3767
  12. ольку                           coef=+0.3704
  13. евод                            coef=+0.3703
  14. льку                            coef=+0.3585
  15.  он.                            coef=+0.3528
  16. чтобы                           coef=+0.3494
  17. тобы                            coef=+0.3494
  18. гда-л                           coef=+0.3477
  19. тобы                            coef=+0.3465
  20. затем  